# Import

In [7]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# Setting

In [8]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 1024
MAX_SEQUENCE_LENGTH = 1024

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [9]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [11]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=1024, max_len=1024)...


Tokenizing: 100%|██████████| 1024/1024 [00:01<00:00, 709.26 examples/s]


2026-02-04T20:23:05.569156+0900 | reset | INFO - Compression lifecycle reset
2026-02-04T20:23:05.570484+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-04T20:23:05.599931+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-04T20:23:05.599931+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-04T20:23:05.605935+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


(1/31): Calibrating: 100%|██████████| 1024/1024 [15:52<00:00,  1.08it/s]

2026-02-04T20:38:58.456494+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 1024 samples


...

2026-02-05T14:48:53.329515+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 1024 samples


2026-02-05T14:48:53.906080+0900 | compress | METRIC - time 0.57s
2026-02-05T14:48:53.907076+0900 | compress | METRIC - error 5916.10
2026-02-05T14:48:53.927847+0900 | compress | METRIC - GPU 0 | usage: 12.81% | total memory: 12 GB
2026-02-05T14:48:53.928843+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-05T14:48:53.932830+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 1024 samples
2026-02-05T14:48:54.306044+0900 | compress | METRIC - time 0.37s
2026-02-05T14:48:54.306044+0900 | compress | METRIC - error 1678.62
2026-02-05T14:48:54.328223+0900 | compress | METRIC - GPU 0 | usage: 12.85% | total memory: 12 GB
2026-02-05T14:48:54.329220+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-05T14:48:54.331422+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 1024 samples
2026-02-05T14:48:54.709751+0900 | compress | METRIC - time 0.38s
2026-02-05T14:48:54.709751+0900 | compress | ME

(31/31): Propagating: 100%|██████████| 1024/1024 [00:01<00:00, 767.32it/s]

2026-02-05T15:01:33.060790+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-05T15:01:33.068531+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


[INFO] GPTQ 완료


# Model Save

In [12]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-05T15:01:33.120060+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:03, 55.52it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [13]:
zip_name = "submit-ver1"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] submit-ver1.zip 생성 중...
[INFO] 생성 완료: submit-ver1.zip
